# TD 2 — Le taux de chômage

**Analyse des données — L3 Économie**

Vous allez travailler sur l'**Enquête Emploi en continu 2024** de l'INSEE : 353 420 personnes interrogées, 83 variables. C'est la seule source permettant de mesurer le chômage au sens du Bureau international du travail.

Le fichier se charge en une ligne, directement depuis le site de l'INSEE.

## Commencez par le dictionnaire

Comme en cours — mais cette fois sur un vrai document de l'INSEE, une trentaine de pages au lieu d'une.

Il s'appelle **« Dictionnaire des codes 2024 »** et se trouve sur la [page du fichier](https://www.insee.fr/fr/statistiques/8632441) comme sur le site du cours. Ouvrez-le maintenant, dans un onglet à côté : vous y reviendrez à chaque question.

Pour chaque variable, vous y retrouverez les rubriques vues en séance :

| Rubrique | Ce qu'elle vous dit |
|---|---|
| Nom | le code court, souvent obscur |
| Libellé | la question posée, ou le concept mesuré |
| Type | caractère ou numérique |
| Modalités | 1 = …, 2 = …, 3 = … |
| **Champ** | **sur qui la variable est-elle renseignée ?** |
| **Non-réponse** | **comment est-elle codée ?** |

Les deux dernières sont celles qu'on oublie de lire, et celles qui font les résultats faux. Ce TD est construit autour d'elles.

> *Exécution → Tout exécuter* avant de démarrer.

## Partie 1 — Charger et inspecter

Le fichier est au format **Parquet** : un format tabulaire moderne, plus compact que le CSV et qui conserve le type de chaque colonne. Il n'y a donc ni séparateur ni encodage à déclarer.

In [ ]:
import pandas as pd
import numpy as np

URL = "https://www.insee.fr/fr/statistiques/fichier/8632441/FD_EEC_2024.parquet"

df = pd.read_parquet(URL)

print(df.shape)
df.head()

**Question 1.** Combien de lignes, combien de colonnes ? De quoi une ligne est-elle la description ?

*Votre réponse :*

In [ ]:
# TODO : affichez la repartition du statut d'activite au sens du BIT
df["________"].value_counts(dropna=False).sort_index()

**Question 2.** Cherchez `ACTEU` dans le dictionnaire — page 4. Trois choses y sont écrites : ses modalités, la variable de pondération à utiliser, et son **champ**.

Quel est ce champ ? Vérifiez-le sur les données en croisant `ACTEU` avec `AGE6`.

*Votre réponse :*

In [ ]:
# TODO : croisez le statut d'activite et l'age
pd.crosstab(df["________"], df["________"], dropna=False)

## Partie 2 — Le taux de chômage

$$\text{taux de chômage} = \frac{\text{chômeurs}}{\text{actifs occupés} + \text{chômeurs}}$$

Commençons par le calcul le plus naturel : compter les personnes.

In [ ]:
# TODO : comptez les individus de chaque statut
n = df["ACTEU"].value_counts()

taux_brut = 100 * n["________"] / (n["________"] + n["________"])
print("Taux de chomage, sans ponderation :", round(taux_brut, 2), "%")

**Question 3.** Quel taux obtenez-vous ?

Cherchez maintenant le taux de chômage publié par l'INSEE pour 2024. L'écart vous paraît-il acceptable ?

*Votre réponse :*

### La pondération

Le dictionnaire l'indique pour **chaque** variable : « variable de pondération à utiliser : `EXTRIAN` ».

L'INSEE le précise aussi sur la page du fichier : pour calculer des statistiques relatives à la population totale de la France, il faut pondérer les données individuelles par le poids individuel. Ce poids indique **combien de personnes de la population chaque répondant représente**.

In [ ]:
# TODO : sommez les poids par statut, au lieu de compter les lignes
p = df.groupby("________")["________"].sum()

taux_pondere = 100 * p["________"] / (p["________"] + p["________"])
print("Taux de chomage, pondere :", round(taux_pondere, 2), "%")

**Question 4.** Comparez les deux taux, puis comparez le taux pondéré au chiffre publié par l'INSEE.

Lequel des deux tombe juste ? Qu'est-ce que cela vous apprend sur les données d'enquête ?

*Votre réponse :*

In [ ]:
# Controle : que represente la somme totale des poids ?
print("Somme des poids :", f"{df['EXTRIAN'].sum():,.0f}".replace(",", " "))

**Question 5.** À quoi ce total correspond-il ? Le chiffre vous paraît-il vraisemblable ?

*Votre réponse :*

## Partie 3 — Deux chiffres officiels

On entend parler de deux statistiques du chômage : les **chômeurs au sens du BIT**, publiés par l'INSEE, et les **demandeurs d'emploi inscrits**, publiés par France Travail.

Ce ne sont pas les mêmes personnes. Le fichier permet de le voir.

In [ ]:
# TODO : croisez le statut BIT et l'inscription comme demandeur d'emploi
# Ponderez le croisement par EXTRIAN
tableau = pd.crosstab(
    df["________"],
    df["________"],
    values=df["EXTRIAN"],
    aggfunc="sum"
)

(tableau / 1000).round(0)     # en milliers

**Question 6.** Le tableau comporte quatre cases intéressantes. Décrivez-les :

- des chômeurs BIT **inscrits** à France Travail ;
- des chômeurs BIT **non inscrits** ;
- des personnes **en emploi** pourtant inscrites ;
- des **inactifs** pourtant inscrits.

Donnez un ordre de grandeur pour chacune, et proposez pour chaque case un exemple de situation concrète.

**Question 7.** Les deux chiffres publiés sont-ils contradictoires ? Lequel utiliseriez-vous pour dimensionner un budget d'indemnisation, et lequel pour comparer la France à l'Allemagne ?

*Vos réponses :*

## Pour ceux qui ont terminé

### Le halo autour du chômage

In [ ]:
# TODO : la variable du halo. Attention a son champ : lisez le dictionnaire.
halo = df.groupby("________")["EXTRIAN"].sum()
(halo / 1000).round(0)

**Question 8.** Combien de personnes le halo représente-t-il, comparé au nombre de chômeurs BIT ?

Ces personnes sont-elles dans votre numérateur, dans votre dénominateur, ou dans aucun des deux ?

**Question 9.** Le dictionnaire précise que `HALOR` n'est renseignée que pour les inactifs. Pourquoi cette restriction est-elle logique ?

*Vos réponses :*

### Décliner le taux

In [ ]:
# TODO : le taux de chomage pondere, par tranche d'age puis par diplome
def taux_par(variable):
    p = df.groupby([variable, "ACTEU"])["EXTRIAN"].sum().unstack(fill_value=0)
    return (100 * p["2"] / (p["1"] + p["2"])).round(2)

print(taux_par("AGE6"))
print()
print(taux_par("DIP7"))

**Question 10.** Commentez les deux profils. Lequel vous surprend le plus ?

*Votre réponse :*

## Avant la séance 3

Vérifiez que votre notebook s'exécute de haut en bas, puis repérez — **sans les traiter** — les variables du fichier qui contiennent des valeurs manquantes.

Le dictionnaire distingue deux situations : « non réponse », codée 9, et « hors champ », codée par une valeur vide. Notez la différence : c'est le sujet de la séance suivante.

---

Supports, données et corrigés : `stefaniamarcassa.github.io/analyse_des_donnees`